In [2]:
import pandas as pd
import pymysql
import gspread 
from oauth2client.service_account import ServiceAccountCredentials
import datetime
import os
from datetime import datetime, timedelta
import glob
from datetime import datetime
import re
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
import pandas as pd
import string  

In [3]:
file_pattern = "ReportFixPriceArchive*.tsv"
file_list = sorted(glob.glob(file_pattern))

for i, file in enumerate(file_list, start=1):
    var_name = f"data{i}"
    
    globals()[var_name] = pd.read_csv(
        file,
        sep='\t',
        header=None,
        quoting=3,
        escapechar=None,
        dtype=str,
        encoding='utf-8',
        on_bad_lines='skip'  # Пропускать строки с ошибками
    )
    
    current_df = globals()[var_name]
    print(f"Прочитан файл {file} -> датафрейм {var_name} (строк: {len(current_df)})")

Прочитан файл ReportFixPriceArchive695.tsv -> датафрейм data1 (строк: 4071403)
Прочитан файл ReportFixPriceArchive696.tsv -> датафрейм data2 (строк: 1618187)
Прочитан файл ReportFixPriceArchive710.tsv -> датафрейм data3 (строк: 874084)


In [4]:
combined_df = pd.concat([data1, data2,data3], ignore_index=True)
print(f"Количество строк в таблице: {len(combined_df)}")
# combined_df = data1
column_names = ['0','1','2','3','4',
    '5','6','7','8','9','10','11','12','13','14','15','16']

combined_df.columns = column_names
combined_df = combined_df[combined_df['9'] == '"RUB"']

df = combined_df.rename(columns={'0': 'Дата обновления',
                         '4': 'Аналог Производитель',
                         '11': 'Цена',
                         '5': 'Доступное количество',
                                '2': 'Аналог Артикул',
                         '16': 'Price ID'
                                })
df = df[['Дата обновления','Аналог Производитель','Аналог Артикул','Цена','Доступное количество','Price ID']].astype(str)
print(f"Количество строк в таблице: {len(df)}")
df =df.astype(str)

Количество строк в таблице: 6563674
Количество строк в таблице: 6507063


In [5]:
df['Дата обновления'] = df['Дата обновления'].astype(str)
df = df[df['Дата обновления'] != 'NaT']
df = df[df['Дата обновления'] != '"']
df = df[df['Price ID'] != 'nan']

print(f"Количество строк в таблице: {len(df)}")

Количество строк в таблице: 6507063


In [6]:
file_name = 'C:/Users/kiekh/OneDrive/Рабочий стол/Index/Рольф/Исходные артикулы (Рольф).xlsx'
sheet_name = 'Исходные SKU'
brands = pd.read_excel(file_name, sheet_name=sheet_name).fillna('').astype(str)
brands['Исходный Артикул'] = brands['Исходный Артикул'].astype(str)
brands['Производитель'] = brands['Производитель'].astype(str)
brands = brands[['Производитель','Исходный Артикул','Наименование исходное','Товарная группа']]
brands = brands[brands['Исходный Артикул'] != '']

brands['Исходный Артикул'] = brands['Исходный Артикул'].astype('str').apply(lambda x: x.replace(' ',''))

file_name = 'C:/Users/kiekh/OneDrive/Рабочий стол/Index/Рольф/Исходные артикулы (Рольф).xlsx'
sheet_name = 'Кроссы'
brands2 = pd.read_excel(file_name, sheet_name=sheet_name).fillna('').astype(str)
brands2['Аналог Артикул'] = brands2['Аналог Артикул'].astype('str').apply(lambda x: x.replace(' ',''))
brands2['Исходный Артикул'] = brands2['Исходный Артикул'].astype('str').apply(lambda x: x.replace(' ',''))

brands = brands2.merge(brands, left_on = 'Исходный Артикул', right_on = 'Исходный Артикул', how = 'left').fillna('')

file_name = 'C:/Users/kiekh/OneDrive/Рабочий стол/Index/Бренд лист/Brand List.xlsx'
sheet_name = 'Brands'
prod = pd.read_excel(file_name, sheet_name=sheet_name).fillna('')
# rrc = rrc[['Артикул','RRP руб. в т.ч. НДС']].rename(columns={'RRP руб. в т.ч. НДС': 'RRP (расчётная)'})
# rrc['Артикул'] = rrc['Артикул'].astype('str').apply(lambda x: x.replace(' ',''))

file_name = 'C:/Users/kiekh/OneDrive/Рабочий стол/Index/Реестр прайс листов/Реестр прайс-листов DM (17.07.2025).xlsx'
sheet_name = 'Price_id'
priceid = pd.read_excel(file_name, sheet_name=sheet_name).astype(str).fillna('')

priceid['Price ID'] = priceid['Price ID'].astype(str)
priceid['Price ID'] = priceid['Price ID'].astype('str').apply(lambda x: x.replace('.0',''))

In [7]:
df.loc[
#     (df['Аналог Производитель'] == 'MANN') 
#     & 
    (df['Аналог Артикул'].str.contains('OC7023', na=False))
]

,Дата обновления,Аналог Производитель,Аналог Артикул,Цена,Доступное количество,Price ID


In [8]:
df['Аналог Производитель'] = df['Аналог Производитель'].astype(str)
df['Аналог Артикул'] = df['Аналог Артикул'].astype(str)

df['Аналог Производитель'] = df['Аналог Производитель'].str.strip('"')
df['Аналог Артикул'] = df['Аналог Артикул'].str.strip('"')
print(f"Количество строк в таблице: {len(df)}")
# df = pd.concat([df, data5], ignore_index=True)
# print(f"Количество строк в таблице: {len(df)}")
# df['Цена'] = df['Цена'].astype('str').apply(lambda x: x.replace('.',','))

df['Доступное количество'] = df['Доступное количество'].astype(str)
df['Доступное количество'] = df['Доступное количество'].astype('str').apply(lambda x: x.replace('.0',''))

df['Price ID'] = df['Price ID'].astype(str)
df['Price ID'] = df['Price ID'].astype('str').apply(lambda x: x.replace('.0',''))
print(f"Количество строк в таблице: {len(df)}")

month_translation = {
    'January': 'Январь',
    'February': 'Февраль',
    'March': 'Март',
    'April': 'Апрель',
    'May': 'Май',
    'June': 'Июнь',
    'July': 'Июль',
    'August': 'Август',
    'September': 'Сентябрь',
    'October': 'Октябрь',
    'November': 'Ноябрь',
    'December': 'Декабрь'
}

df['Дата обновления'] = pd.to_datetime(df['Дата обновления'], errors='coerce')

df['Квартал'] = df['Дата обновления'].dt.quarter
df['Номер месяца'] = df['Дата обновления'].dt.month  # Номер месяца (1-12)
df['Месяц'] = df['Дата обновления'].dt.month_name().map(month_translation)
df['Неделя'] = df['Дата обновления'].dt.isocalendar().week
df['День'] = df['Дата обновления'].dt.day
df['Год'] = df['Дата обновления'].dt.year

print(f"Количество строк в таблице: {len(df)}")

df['Квартал'] = df['Квартал'].astype(str)
df['Квартал'] = df['Квартал'].astype('str').apply(lambda x: x.replace('.0',''))

df['Неделя'] = df['Неделя'].astype(str)
df['Неделя'] = df['Неделя'].astype('str').apply(lambda x: x.replace('.0',''))


print(f"Количество строк в таблице: {len(df)}")

Количество строк в таблице: 6507063
Количество строк в таблице: 6507063
Количество строк в таблице: 6507063
Количество строк в таблице: 6507063


In [9]:
dff = df.astype(str)

dff = dff.merge(brands, left_on = 'Аналог Артикул', right_on = 'Аналог Артикул', how = 'left').fillna('')
print(f"Количество строк в таблице: {len(dff)}")
dff = dff.merge(prod, left_on = 'Аналог Производитель', right_on = 'Аналог Производитель', how = 'left').fillna('')
print(f"Количество строк в таблице: {len(dff)}")
dff = dff.merge(priceid, left_on = 'Price ID', right_on = 'Price ID', how = 'left').fillna('')
print(f"Количество строк в таблице: {len(dff)}")
def check_analog(row):
    if row['Аналог Артикул'] == row['Исходный Артикул']:
        return 'Genuine'
    else:
        return 'Analog'

# Применяем функцию к DataFrame
dff['Тип артикула'] = dff.apply(check_analog, axis=1)

dff = dff.astype(str)

Количество строк в таблице: 13455130
Количество строк в таблице: 13455130
Количество строк в таблице: 13455130


In [18]:
df = dff


df = df[df['Аналог Артикул'] != '']
df = df[df['Исходный Артикул'] != '']
df = df[df['Цена'] != '']
df = df.drop(columns=['Аналог Производитель', 'Аналог Артикул'])
print(f"Количество строк в таблице: {len(df)}")

df = df[df['Price ID'] != '']

df = df[df['Вид поставок'] != 'Служебный']
df = df[df['Вид поставок'] != '']
df = df[df['Вид поставок'] != 'Транзит']

df = df[df['Наименование исходное'] != '']
df = df[df['Цена'] != '']
# df = df[df['Производитель'] == 'MANN-FILTER']

print(f"Количество строк в таблице: {len(df)}")
df['Доступное количество'] = df['Доступное количество'].astype('int64')


df['Цена'] = df['Цена'].astype(float)
df = df[df['Цена'] < 1000000]
df = df[df['Цена'] > 2]

group_cols_max = ['Неделя', 'Производитель', 'Исходный Артикул', 'Price ID']
group_cols_max2 = ['Дата обновления', 'Производитель', 'Исходный Артикул', 'Price ID']
group_cols_max3 = ['Дата обновления', 'Производитель', 'Исходный Артикул', 'Price ID']

group_cols_rsp = ['Неделя', 'Исходный Артикул']

df['Цена (средняя)'] = df.groupby(group_cols_max3)['Цена'].transform('mean')
df['Цена (медианная)'] = df.groupby(group_cols_max3)['Цена'].transform('median')
df['RRP, руб.'] = df.groupby(group_cols_rsp)['Цена'].transform('median')
df['Цена (макс)'] = df.groupby(group_cols_max3)['Цена'].transform('max')
df['Цена (мин)'] = df.groupby(group_cols_max3)['Цена'].transform('min')
df['DMaxStock'] = df.groupby(group_cols_max2 )['Доступное количество'].transform('max')
df['WmaxStock'] = df.groupby(group_cols_max )['Доступное количество'].transform('max')
df['WminStock'] = df.groupby(group_cols_max)['Доступное количество'].transform('min')
df = df.drop_duplicates()
print(f"Количество строк в таблице: {len(df)}")

df = df[[
         'Дата обновления',
         'Квартал',
         'Месяц',
         'Неделя',
         'День',
         'Исходный Артикул',       
         'Производитель',          
         'Наименование исходное',
         'Товарная группа',
#          'OEM/IAM',
#          'Категория IAM',
         'Тип артикула',
         'Цена (макс)',
         'Цена (мин)',
         'Цена (средняя)',
         'Цена (медианная)',
         'RRP, руб.',
         'DMaxStock',
         'WmaxStock',
         'WminStock',
         'Price ID',
         'Вид поставок',
         'Тип поставщика',
         'Город',
         'Регион',
         'Округ','Год']]

df = df.drop_duplicates()

df['Цена (средняя)'] = df['Цена (средняя)'].astype(float)
df['RRP, руб.'] = df['RRP, руб.'].astype(float)

df['RRP Gap, %'] = (df['Цена (макс)'] - df['RRP, руб.']) / df['RRP, руб.']


print(f"Количество строк в таблице: {len(df)}")
df['Цена2'] = df['Цена (макс)'].astype(int)

max_price = df['Цена2'].max()
step = 500
bins_price = list(range(0, 12500 + step, step)) + [float('inf')]  # Добавляем +inf для последнего интервала

# Буквенные метки: A, B, ..., Y, Z
letters = list(string.ascii_uppercase)  # A, B, ..., Z
num_bins = len(bins_price) - 1  # Количество интервалов

# Генерация меток
labels_price = []
for i in range(num_bins):
    if i < len(letters) - 1:  # Для A, B, ..., Y
        labels_price.append(f"{letters[i]}. {bins_price[i]}-{bins_price[i+1]}")
    else:  # Для Z и остальных (если интервалов >26)
        labels_price.append(f"Z. {bins_price[i]}+")

# Создание столбца с диапазонами
df['Диапазон цены'] = pd.cut(
    df['Цена2'],
    bins=bins_price,
    labels=labels_price,
    right=False,  # [0, 100), [100, 200), ...
    include_lowest=True
)
bins_stock = [0, 1, 2, 10, 50, 100, 1000, 5000,10000, float('inf')]
labels_stock = [
    '0. Нет в наличии',  # 0 шт.
    '1. 1 шт.',          # 1 шт.
    '2. 2-9 шт.',        # от 2 до 9 шт.
    '3. 10-49 шт.',      # от 10 до 49 шт.
    '4. 50-99 шт.',      # от 50 до 99 шт.
    '5. 100-999 шт.',    # от 100 до 999 шт.
    '6. 1000-4999 шт.',  # от 1000 до 2999 шт.
    '7. 5000-9999 шт.',  # от 3000 до 4999 шт.
    '8. 10000+ шт.',  # от 3000 до 4999 шт.

]
df['Диапазон количества'] = pd.cut(
    df['WmaxStock'],
    bins=bins_stock,
    labels=labels_stock,
    right=False  # Например, 4 попадет в B. 4-9, а не в A. <4
)
df = df[[
         'Дата обновления',
         'Квартал',
         'Месяц',
         'Неделя',
         'День',
         'Исходный Артикул',       
         'Производитель',      
         'Наименование исходное',
         'Товарная группа',
#          'OEM/IAM',
#          'Категория IAM',
         'Диапазон цены',        
         'Тип артикула',
         'Цена (макс)',
         'Цена (мин)',
         'Цена (средняя)',
         'Цена (медианная)',
         'RRP, руб.',
         'RRP Gap, %',
         'Диапазон количества',
         'DMaxStock',
         'WmaxStock',
         'WminStock',
         'Price ID',
         'Вид поставок',
         'Тип поставщика',
         'Город',
         'Регион',
         'Округ','Год']]
df = df.drop_duplicates()


print(f"Количество строк в таблице: {len(df)}")

Количество строк в таблице: 13281410
Количество строк в таблице: 12907702
Количество строк в таблице: 12284821
Количество строк в таблице: 1307102
Количество строк в таблице: 1307102


In [19]:
dfttt = df
for week_num in dfttt['Год'].unique():
    # Создаем имя переменной (df14, df15 и т.д.)
    var_name = f'dfttt{week_num}'
    
    # Создаем датафрейм для текущей недели
    globals()[var_name] = dfttt[dfttt['Год'] == week_num]
#     globals()[var_name].to_sql(
#         name='brake_discs_aut',  # Имя таблицы для записи
#         con=engine,
#         if_exists='append',  # 'append' - добавить, 'replace' - перезаписать
#         index=False
#         )
    # Выводим статистику
    print(f"Количество строк в таблице dfttt{week_num}: {len(globals()[var_name])}")

Количество строк в таблице dfttt2025: 596221
Количество строк в таблице dfttt2024: 710881


In [20]:
# max_rows = 1_048_576  # Максимальное поддерживаемое значение (например, для Excel)

# # Обрезаем обе таблицы до max_rows (если они больше)
# dfttt1 = dfttt1.iloc[:max_rows] if len(dfttt1) > max_rows else dfttt1
# dfttt2 = dfttt2.iloc[:max_rows] if len(dfttt2) > max_rows else dfttt2

salary_sheets = {
'WStats 2025':dfttt2025,
'WStats 2024':dfttt2024,
}



In [21]:
writer = pd.ExcelWriter('C:/Users/kiekh/OneDrive/Рабочий стол/Index/Рольф/Автомир (дни).xlsx', engine='xlsxwriter')
# writer1 = pd.ExcelWriter('C:/Users/kiekh/OneDrive/Рабочий стол/Тачки/Диски тормозные AUT 1.xlsx', engine='xlsxwriter')
# writer2 = pd.ExcelWriter('C:/Users/kiekh/OneDrive/Рабочий стол/Тачки/Диски тормозные AUT 2.xlsx', engine='xlsxwriter')

In [22]:
for sheet_name in salary_sheets.keys():
    salary_sheets[sheet_name].to_excel(writer, sheet_name=sheet_name, index=False)

writer.save()

C:\Users\kiekh\AppData\Local\Temp\ipykernel_14212\562198196.py:4: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()


In [26]:
for sheet_name in salary_sheets1.keys():
    salary_sheets1[sheet_name].to_excel(writer1, sheet_name=sheet_name, index=False)

writer1.save()

C:\Users\kiekh\AppData\Local\Temp\ipykernel_32\2431875120.py:4: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer1.save()


In [14]:
for sheet_name in salary_sheets2.keys():
    salary_sheets2[sheet_name].to_excel(writer2, sheet_name=sheet_name, index=False)

writer2.save()

C:\Users\kiekh\AppData\Local\Temp\ipykernel_11872\2428425300.py:4: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer2.save()
